In [ ]:
# Imports and input PDF path setup
import re
import json
from pathlib import Path

import pdfplumber

PDF_PATH = Path(
    r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\Pile Foundation_Part 2.pdf"
)

print(f"Input PDF: {PDF_PATH}")

Input PDF: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\Pile Foundation_Part 2.pdf


In [ ]:
# Read PDF page-by-page and collect cleaned text + extracted tables
pages_data = []

with pdfplumber.open(str(PDF_PATH)) as pdf:
    total = len(pdf.pages)
    print(f"[INFO] '{PDF_PATH.name}' - {total} pages")

    for i, page in enumerate(pdf.pages, start=1):
        print(f"  -> Page {i}/{total}...", end="\r")

        # Clean page text
        raw_text = page.extract_text(x_tolerance=3, y_tolerance=3) or ""
        text = raw_text.replace("\u00a0", " ")
        text = re.sub(r"\(cid:\d+\)", "", text)
        text = re.sub(r"\n\s*\n+", "\n\n", text)
        lines = [line.rstrip() for line in text.split("\n")]
        text = "\n".join(lines).strip()

        # Extract tables and convert them to markdown-friendly format
        tables_found = []
        for table in page.extract_tables():
            if not table:
                continue

            cleaned_rows = [
                [str(cell).strip() if cell is not None else "" for cell in row]
                for row in table
            ]

            md_lines = []
            for row_idx, row in enumerate(cleaned_rows):
                md_lines.append(" | ".join(row))
                if row_idx == 0:
                    md_lines.append(" | ".join(["---"] * len(row)))

            tables_found.append(
                {
                    "rows": cleaned_rows,
                    "markdown": "\n".join(md_lines),
                }
            )

        pages_data.append(
            {
                "page": i,
                "text": text,
                "tables": tables_found,
                "has_tables": bool(tables_found),
            }
        )

print()

[INFO] 'Pile Foundation_Part 2.pdf' - 15 pages
  -> Page 15/15...


In [ ]:
# Build one document object with page text and table blocks
parts = []
for p in pages_data:
    if p["text"]:
        parts.append(f"[PAGE {p['page']}]\n{p['text']}")
    for t in p["tables"]:
        parts.append(f"[TABLE - PAGE {p['page']}]\n{t['markdown']}")

doc = {
    "source": str(PDF_PATH),
    "file_name": PDF_PATH.name,
    "total_pages": total,
    "pages": pages_data,
    "full_text": "\n\n".join(parts),
}

print(f"Doc assembled: {doc['file_name']} ({doc['total_pages']} pages)")

Doc assembled: Pile Foundation_Part 2.pdf (15 pages)


In [ ]:
# Flatten page-wise data into embedding-ready records
records = []

for page in doc["pages"]:
    base = {
        "source": doc["source"],
        "file_name": doc["file_name"],
        "page": page["page"],
    }

    if page["text"]:
        records.append({**base, "kind": "text", "text": page["text"]})

    for idx, table in enumerate(page["tables"], start=1):
        records.append(
            {
                **base,
                "kind": "table",
                "table_index": idx,
                "text": table["markdown"],
                "rows": table["rows"],
            }
        )

# Quick summary counts
text_count = sum(1 for r in records if r["kind"] == "text")
table_count = sum(1 for r in records if r["kind"] == "table")
print(f"Records - total: {len(records)} | text: {text_count} | tables: {table_count}")

Records - total: 15 | text: 15 | tables: 0


In [ ]:
# Preview sample records and save output JSON
for r in records[:2]:
    print(f"[{r['kind'].upper()} | Page {r['page']}]\n{r['text'][:300]}\n...\n")


[TEXT | Page 1]
06-03-2026
Meyerhof’s Method for Estimating Q
p
• The point bearing or end bearing (Q or Q )
p b
• 𝑄  𝐴 𝑞  𝐴 𝑞𝑁∗  𝐴 𝑞

• where, 𝑞  0.5𝑝 𝑁∗tan

p = atmospheric pressure
a
= effective friction angle of the bearing
stratum
Figure 21
51
Figure 22
Source:Meyerhof,G.G.
(1976).“BearingCapacity
andSettl
...

[TEXT | Page 2]
06-03-2026
Frictional Resistance Q in sand
s
• It has been observed that the nature of variation of f in the field is
approximately as shown in Figure 23. The unit skin friction increases
with depth more or less linearly to a depth of Land remains constant
thereafter. The magnitude of the critical 
...

[SAVED] C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\Pile Foundation_Part 2.json
